In [ ]:
import time
import numpy as np
import json
import pickle
import websocket
import socket
import os

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# Resolve project root (works from notebooks_deap/ folder)
_cwd = os.path.abspath(os.getcwd())
_candidates = [_cwd] + [os.path.abspath(os.path.join(_cwd, *([".."]*i))) for i in range(1, 6)]
PROJECT_ROOT = next((p for p in _candidates if os.path.isdir(os.path.join(p, "data", "DEAP", "data_preprocessed_python"))), _cwd)

BASE_DEAP_PATH = os.path.join(PROJECT_ROOT, "data", "DEAP", "data_preprocessed_python")
SERVER_URI = "ws://localhost:65432"

# 16 channel indices from DEAP's 32 channels (standard 10-20 subset)
CHANNELS_IDX = [0, 2, 3, 6, 7, 10, 11, 13, 16, 19, 20, 24, 25, 28, 29, 31]

print(f"PROJECT_ROOT:   {PROJECT_ROOT}")
print(f"BASE_DEAP_PATH: {BASE_DEAP_PATH}")
print(f"SERVER_URI:     {SERVER_URI}")

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def load_data(subject_id):
    file_path = os.path.join(BASE_DEAP_PATH, f"s{subject_id}.dat")
    print(f"📂 Loading data for S{subject_id}...")
    try:
        with open(file_path, 'rb') as f:
            content = pickle.load(f, encoding='latin1')
        return content['data'], content['labels']
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        return None, None


def get_latest_command(ws, blocking=False):
    """Read commands from the WebSocket (Play, Pause, Resume)."""
    cmd = None
    if blocking:
        ws.sock.settimeout(None)
    else:
        ws.sock.settimeout(0.001)

    try:
        while True:
            msg = ws.recv()
            try:
                data = json.loads(msg)
                if isinstance(data, dict) and data.get("type") in [
                    "cmd_start_stream", "cmd_pause_stream", "cmd_resume_stream"
                ]:
                    cmd = data
                    if blocking:
                        return cmd
            except (json.JSONDecodeError, ValueError):
                pass
    except (socket.timeout, websocket.WebSocketTimeoutException):
        pass
    except Exception as e:
        raise e

    return cmd


# ─────────────────────────────────────────────────────────────────────────────
# MAIN STREAMING LOOP
# ─────────────────────────────────────────────────────────────────────────────
def stream_deap():
    ws = websocket.WebSocket()
    headers = {"User-Agent": "Mozilla/5.0 (Compatible)"}

    current_subject = None
    data = None
    labels = None

    try:
        print(f"🔌 Connecting to {SERVER_URI}...")
        ws.connect(SERVER_URI, header=headers, timeout=20)
        print("🟢 CONNECTED! DEAP streamer ready.\n")

        state = "WAITING"
        active_cmd = None

        while True:
            # ── STATE 1: Wait for Play command from UI ──
            if state == "WAITING":
                print("⏳ Waiting for 'Play' command from the Web Interface...")
                active_cmd = get_latest_command(ws, blocking=True)
                if active_cmd["type"] == "cmd_start_stream":
                    state = "PLAYING"

            # ── STATE 2: Stream the requested trial ──
            if state == "PLAYING":
                subj = active_cmd["subject"]
                trial_idx = active_cmd["trial"]

                if subj != current_subject or data is None:
                    data, labels = load_data(subj)
                    current_subject = subj

                if data is None:
                    state = "WAITING"
                    continue

                val_true = labels[trial_idx][0]
                aro_true = labels[trial_idx][1]

                print(f"\n▶️ START STREAM -> S{current_subject} | TRIAL {trial_idx + 1}/40")
                print(f"   Target: Valence={val_true:.2f}, Arousal={aro_true:.2f}")

                ws.send(json.dumps({
                    "type": "stream_info",
                    "true_valence": float(val_true),
                    "true_arousal": float(aro_true)
                }))

                trial_signal = data[trial_idx][CHANNELS_IDX, :]
                total_samples = trial_signal.shape[1]
                cursor = 0
                interrupted = False

                while cursor < total_samples:
                    # Check for UI commands (Pause, Replay)
                    new_cmd = get_latest_command(ws, blocking=False)
                    if new_cmd:
                        if new_cmd["type"] == "cmd_start_stream":
                            print("\n⚠️ Interrupted: new Play/Replay command received!")
                            active_cmd = new_cmd
                            interrupted = True
                            break
                        elif new_cmd["type"] == "cmd_pause_stream":
                            print("\n⏸️ Paused... Waiting for Resume.")
                            while True:
                                resume_cmd = get_latest_command(ws, blocking=True)
                                if resume_cmd["type"] == "cmd_resume_stream":
                                    print("▶️ Resuming stream!")
                                    break
                                elif resume_cmd["type"] == "cmd_start_stream":
                                    active_cmd = resume_cmd
                                    interrupted = True
                                    break
                            if interrupted:
                                break

                    # Send signal chunk (16 samples = 0.125s at 128Hz)
                    chunk = trial_signal[:, cursor:cursor+16]
                    ws.send(json.dumps(chunk.tolist()))

                    # Send progress update
                    pct = int((cursor / total_samples) * 100)
                    ws.send(json.dumps({"type": "progress", "percent": pct}))

                    cursor += 16
                    time.sleep(0.125)

                if not interrupted:
                    ws.send(json.dumps({"type": "stream_end"}))
                    print("⏹️ Trial completed successfully.")
                    state = "WAITING"

    except KeyboardInterrupt:
        print("\nManual stop.")
    finally:
        ws.close()


if __name__ == "__main__":
    stream_deap()

PROJECT_ROOT:   c:\Users\PC\Desktop\EEG_GraphAttentionNetwork
BASE_DEAP_PATH: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\data_preprocessed_python
SERVER_URI:     ws://localhost:65432
🔌 Connecting to ws://localhost:65432...
🟢 CONNECTED! DEAP streamer ready.

⏳ Waiting for 'Play' command from the Web Interface...
📂 Loading data for S01...

▶️ START STREAM -> S01 | TRIAL 1/40
   Target: Valence=7.71, Arousal=7.60

⚠️ Interrupted: new Play/Replay command received!
📂 Loading data for S02...

▶️ START STREAM -> S02 | TRIAL 1/40
   Target: Valence=9.00, Arousal=5.03

⚠️ Interrupted: new Play/Replay command received!

▶️ START STREAM -> S02 | TRIAL 1/40
   Target: Valence=9.00, Arousal=5.03

⚠️ Interrupted: new Play/Replay command received!

▶️ START STREAM -> S02 | TRIAL 1/40
   Target: Valence=9.00, Arousal=5.03

⚠️ Interrupted: new Play/Replay command received!

▶️ START STREAM -> S02 | TRIAL 3/40
   Target: Valence=9.00, Arousal=9.00

⚠️ Interrupted: new Play/Replay command re